In [5]:
import ctypes
import math

class DynamicArray:
    def __init__(self):
        self.__size = 0
        self.__capacity = 1
        self.__array = self.__allocate(self.__capacity)

    def __allocate(self, capacity):
        return (ctypes.py_object * capacity)()

    def len(self):
        return self.__size

    def iter_seq(self):
        for i in range(self.__size):
            yield self.__array[i]

    def build(self, X):
        self.__size = len(X)

        if self.__size == 0:
            self.__capacity = 1
        else:
            self.__capacity = 2**math.ceil(math.log2(self.__size))
        
        self.__array = self.__allocate(self.__capacity)
        
        for i in range(self.__size):
            self.__array[i] = X[i]

    def get_at(self, i):
        if i < 0 or i >= self.__size:
            raise IndexError

        return self.__array[i]

    def set_at(self, i, x):
        if i < 0 or i >= self.__size:
            raise IndexError

        self.__array[i] = x

    def __resize(self, new_capacity, copy_count):
        temp_array = self.__array
        self.__array = self.__allocate(new_capacity)
        for i in range(copy_count):
            self.__array[i] = temp_array[i]
        self.__capacity = new_capacity

    def __grow_if_needed(self):
        if self.__size == self.__capacity:
            self.__resize(2**math.ceil(math.log2(self.__size + 1)), self.__size)

    def __shrink_if_needed(self):
        if self.__capacity > 1 and (self.__size - 1) * 4 <= self.__capacity:
            self.__resize(max(1, self.__capacity // 2), self.__size - 1)

    def insert_at(self, i, x):
        if i < 0 or i > self.__size:
            raise IndexError
        
        self.__grow_if_needed()
        for j in range(self.__size, i, -1):
            self.__array[j] = self.__array[j-1]
        self.__array[i] = x
        self.__size += 1

    def delete_at(self, i):
        if i < 0 or i >= self.__size:
            raise IndexError
            
        value = self.__array[i]
        for j in range(i, self.__size -1):
            self.__array[j] = self.__array[j+1]
        self.__array[self.__size - 1] = None
        self.__shrink_if_needed()
        self.__size -= 1

        return value

    def insert_first(self, x):
        self.insert_at(0, x)

    def insert_last(self, x):
        self.insert_at(self.__size, x)
        
    def delete_first(self):
        return self.delete_at(0)

    def delete_last(self):
        return self.delete_at(self.__size - 1)



In [6]:
arr = DynamicArray()

# Build, len, iter_seq
arr.build([10, 20, 30, 40])
assert arr.len() == 4
assert list(arr.iter_seq()) == [10, 20, 30, 40]

# Get and set
assert arr.get_at(1) == 20
arr.set_at(1, 25)
assert list(arr.iter_seq()) == [10, 25, 30, 40]

# Insert first
arr.insert_first(5)
assert list(arr.iter_seq()) == [5, 10, 25, 30, 40]

# Insert last
arr.insert_last(50)
assert list(arr.iter_seq()) == [5, 10, 25, 30, 40, 50]

# Insert at
arr.insert_at(2, 15)
assert list(arr.iter_seq()) == [5, 10, 15, 25, 30, 40, 50]

# Delete first
assert arr.delete_first() == 5
assert list(arr.iter_seq()) == [10, 15, 25, 30, 40, 50]

# Delete last
assert arr.delete_last() == 50
assert list(arr.iter_seq()) == [10, 15, 25, 30, 40]

# Delete at
assert arr.delete_at(1) == 15
assert list(arr.iter_seq()) == [10, 25, 30, 40]

# Growth
arr.build([])

for i in range(1000):
    arr.insert_last(i)

assert arr.len() == 1000
assert list(arr.iter_seq()) == list(range(1000))

# Shrinking
for i in range(999, -1, -1):
    assert arr.delete_last() == i

assert arr.len() == 0
assert list(arr.iter_seq()) == []

# Reuse after emptying
arr.insert_last(100)
assert arr.len() == 1
assert arr.get_at(0) == 100

print("All tests passed!")

All tests passed!
